## Demonstration and Testing of Time-Varying Mean capabilities

In [ ]:
import numpy as np
from datetime import datetime, timedelta
from stonesoup.types.groundtruth import GroundTruthPath, GroundTruthState
from stonesoup.models.transition.linear import RandomWalk
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.driver import  AlphaStableNSMDriver
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.models.transition.linear import ConstantVelocity

In [ ]:
mu_model='CV'
run_inference=True
avoid_subintervals=True
if avoid_subintervals:
    mu_W_state=None
show_1D=True
save_1D=True
plot_tracks=True
if plot_tracks:
    uncertainty=True
    particle=False
    plot_particle_paths=False

In [ ]:
colors_dict={}
colors_dict['truth']='blue'
colors_dict['track']='#00CC96'
colors_dict['culled_track']='#FFA15A'
colors_dict['RTS_track']='#B6E880'
colors_dict['CK_track']='#AB63FA'

In [ ]:
seed = 3 # Random seem for reproducibility

In [ ]:
start_time = datetime.now().replace(microsecond=0)

#time-varying skew parameters
sigma_mu= 0.015
q=sigma_mu**2
if mu_model=='RW':
    initial_mu_W_x= +0.1
    initial_mu_W_y= +0.05
    RW_mu_driver= RandomWalk(noise_diff_coeff=q,seed=seed) #1D GRW
    mu_driver = RW_mu_driver
elif mu_model=='CV':
    q=(sigma_mu**2)/1e4
    initial_mu_W_x= np.array([[0.1],[-0.00]])
    initial_mu_W_y= np.array([[0.5],[-0.001]])
    CV_mu_driver=ConstantVelocity(noise_diff_coeff=q,seed=seed)
    mu_driver =CV_mu_driver

# Driving process parameters and drivers
sigma_W2 = 1e-5**2
alpha = 0.9
c=10

driver_x = AlphaStableNSMDriver(mu_W=initial_mu_W_x, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver, mu_W_state=mu_W_state)
driver_y = AlphaStableNSMDriver(mu_W=initial_mu_W_y, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver, mu_W_state=mu_W_state)

# transition params and model
theta=0.05
num_steps =300
number_particles = 5000
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

#measurement params and model
k_v=2800e4
if mu_model=='CV':
    k_v/=2
measurement_model = LinearGaussian(
    ndim_state=4,  # Number of state dimensions (position and velocity in 2D)
    mapping=(0, 2),  # Mapping measurement vector index to state index
    noise_covar=np.array([[sigma_W2*k_v**2, 0],  # Covariance matrix for Gaussian PDF
                          [0, sigma_W2*k_v**2]])
    )

In [ ]:
timesteps = [start_time]
x_mu_data=[]
y_mu_data=[]
x_mu_vel=[]
y_mu_vel=[]

truth = GroundTruthPath([GroundTruthState([0, 1, 0, 1], timestamp=timesteps[0])])

for k in range(num_steps):
    timesteps.append(start_time+timedelta(seconds=1*(k+1)))  # add next timestep to list of timesteps
    truth.append(GroundTruthState(
        transition_model.function(truth[k], noise=True, time_interval=timedelta(seconds=1),seed=seed),
        timestamp=timesteps[k+1]))
    x_mu_data.append(transition_model.mu_W[0,0,0])
    y_mu_data.append(transition_model.mu_W[0,0,1])
    if mu_model=='CV':
        x_mu_vel.append(transition_model.mu_W[0,1,0])
        y_mu_vel.append(transition_model.mu_W[0,1,1])
    # need to explain in tutorial that shape is n x m x num_drivers, 
    # where for us each driver is m=1 but we have 2 of them in the combined model

In [ ]:
from pathlib import Path
from stonesoup.plotter import AnimatedPlotterly, Plotterly,  Dimension
folder_path=rf"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\TimeVaryingPlots"

In [ ]:
from plotly.subplots import make_subplots

# your colour dict
colors_dict = {
    'truth':        'blue',
    'track':        '#00CC96',
    'culled_track': '#FFA15A',
    'RTS_track':    '#B6E880',
    'CK_track':     '#AB63FA'
}

axis_label_list = ["X1(t)","dX1(t)_dt","X2(t)","dX2(t)_dt"]
particle_plotter_dict = {}

for i, label in enumerate(axis_label_list):
    # select the right mu data & estimates
    if i <= 1:
        data           = x_mu_data
    else:
        data           = y_mu_data

    # for CV only, pick velocity
    if mu_model == 'CV':
        if i <= 1:
            vel_data           = x_mu_vel
        else:
            vel_data           = y_mu_vel

    def new_plotter(label):
        p = Plotterly(autosize=False, width=1500, height=800,
                      dimension=Dimension.ONE, axis_labels=[label])
        cw = p.fig.layout.colorway
        p.fig = make_subplots(specs=[[{"secondary_y": True}]])
        p.fig.update_yaxes(
            secondary_y=False,
            title_text=f"{label}",
            mirror=True,
            ticks='outside',
            showline=True,
            linecolor='black',
            title=dict(text=label, font=dict(size=20))
            )
        p.fig.update_yaxes(
            secondary_y=True,
            title_text="mu_W",
            mirror=True,
            ticks='outside',
            showline=True,
            linecolor='black',
            gridcolor='lightgrey',
            title=dict(text="Time", font=dict(size=20))
        )
        p.fig.update_layout(
                plot_bgcolor='white',
        legend=dict(
                    font=dict(size=15),       # Make the legend font larger
                    orientation='h',
                    # xanchor="auto",         # Center the legend
                    # yanchor="auto",           # Align the legend to the bottom of the plot
                    bordercolor="Black",
                    borderwidth=3,
                    y=0.055,                   # Position it above the graph
                    # x=0.6                    # Center it horizontally
                ),)
        p.fig.update_xaxes(
                title=dict(text="Time", font=dict(size=20)),
                mirror=True,
                ticks='outside',
                showline=True,
                linecolor='black',
                gridcolor='lightgrey',
            )

        p.fig.layout.colorway = cw
        return p

    # -----------------------------------------------------------------
    # 1) Simulation: TRUE + µ on secondary  (x dims only: i=0,2)
    # -----------------------------------------------------------------
    if i in (0, 2):
        p = new_plotter(label)
        p.plot_ground_truths(truth, [i], mode="lines",
                             name=f'{label} groundtruth',
                             line=dict(color=colors_dict['truth'], width=3))
        p.fig.add_scatter(x=timesteps, y=np.array(data),
                          name='µ groundtruth', secondary_y=True,
                          line=dict(color='red', width=3))
        p.fig.add_scatter(x=timesteps, y=np.zeros_like(data),
                          name='µ=0', secondary_y=True,
                          line=dict(color='gray', dash='dash'))

        p.fig.update_yaxes(secondary_y=False)
        p.fig.update_yaxes(title_text="µ",secondary_y=True)

        fname = f"{mu_model}_1D_{label}_sim_mu.html"
        p.fig.write_html(Path(folder_path)/fname)
        particle_plotter_dict[f"{label}_sim_mu"] = p
        p.fig.show()
    # -----------------------------------------------------------------
    # 2) Simulation: TRUE + µ on secondary  (velocity dims: i=1,3)
    # -----------------------------------------------------------------
    if i in (1, 3):
        p = new_plotter(label)
        p.plot_ground_truths(truth, [i], mode="lines",
                          name=f'{label} groundtruth',
                          line=dict(color=colors_dict['truth'], width=3))
        p.fig.add_scatter(x=timesteps, y=np.array(data),
                          name='µ groundtruth', secondary_y=True,
                          line=dict(color='red', width=3))
        p.fig.add_scatter(x=timesteps, y=np.zeros_like(data),
                          name='µ=0', secondary_y=True,
                          line=dict(color='gray', dash='dash'))
        p.fig.update_yaxes(secondary_y=False)
        p.fig.update_yaxes(title_text="µ", secondary_y=True)
        fname = f"{mu_model}_1D_{label}_sim_mu_vel.html"
        p.fig.write_html(Path(folder_path)/fname)
        particle_plotter_dict[f"{label}_sim_mu_vel"] = p
        p.fig.show()

In [ ]:
from stonesoup.types.array import StateVectors
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type


# Sample from the prior Gaussian distribution, input MxN 
if mu_model=='RW':
    mu_prior_x = np.atleast_2d(multivariate_normal.rvs(initial_mu_W_x,
                                np.diag([0.1*q]),
                                size=number_particles))
    mu_prior_y = np.atleast_2d(multivariate_normal.rvs(initial_mu_W_y,
                                np.diag([0.1*q]),
                                size=number_particles))
    mu_driver = RW_mu_driver #1D GRW
elif mu_model=='CV':
    mu_prior_x = multivariate_normal.rvs(initial_mu_W_x.flatten(),
                                np.diag([q,0]),
                                size=number_particles).T
    mu_prior_y = multivariate_normal.rvs(initial_mu_W_y.flatten(),
                                np.diag([q,0]),
                                size=number_particles).T
    mu_driver= CV_mu_driver

if avoid_subintervals:
    mu_W_state=None
else:
    mu_W_state=True

driver_x = AlphaStableNSMDriver(mu_W=mu_prior_x, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver,mu_W_state=mu_W_state)
driver_y = AlphaStableNSMDriver(mu_W=mu_prior_y, sigma_W2=sigma_W2, seed=seed, c=c, alpha=alpha, mu_W_transition_model=mu_driver,mu_W_state=mu_W_state)
langevin_x = LevyLangevin(driver=driver_x, damping_coeff=theta)
langevin_y = LevyLangevin(driver=driver_y, damping_coeff=theta)
transition_model = CombinedLinearLevyTransitionModel([langevin_x, langevin_y])

In [ ]:
if run_inference:
    from stonesoup.types.detection import Detection
    measurements = []
    for state in truth:
        measurement = measurement_model.function(state, noise=True)
        measurements.append(Detection(measurement,
                                    timestamp=state.timestamp,
                                    measurement_model=measurement_model))
        
    from stonesoup.predictor.particle import MarginalisedParticlePredictor
    from stonesoup.resampler.particle import SystematicResampler 
    from stonesoup.updater.particle import MarginalisedParticleUpdater

    predictor = MarginalisedParticlePredictor(transition_model=transition_model)
    resampler = SystematicResampler()
    updater = MarginalisedParticleUpdater(measurement_model, resampler)

    from stonesoup.types.state import MarginalisedParticleState
    from stonesoup.types.hypothesis import SingleHypothesis
    from stonesoup.types.track import Track

    # Sample from the prior Gaussian distribution
    states = multivariate_normal.rvs(np.array([0, 1, 0, 1]),
                                    np.diag([1., 1., 1., 1.]),
                                    size=number_particles)
    covars = np.stack([np.eye(4) * 10 for i in range(number_particles)], axis=2) # (M, M, N)
    # Create prior particle state.
    prior = MarginalisedParticleState(
        state_vector=StateVectors(states.T),
        covariance=covars,
        weight=np.array([Probability(1/number_particles)]*number_particles),
                        timestamp=start_time-timedelta(seconds=1))

    track = Track()
    x_mu_estimate = []
    y_mu_estimate = []
    x_mu_vel_estimate = []
    y_mu_vel_estimate = []
    for measurement in measurements:
        prediction = predictor.predict(prior, timestamp=measurement.timestamp)
        hypothesis = SingleHypothesis(prediction, measurement)
        post = updater.update(hypothesis)
        track.append(post)
        prior = track[-1]   
        x_mu_estimate.append(np.mean(transition_model.mu_W[0,0,0]))
        y_mu_estimate.append(np.mean(transition_model.mu_W[0,0,1]))
        if mu_model=='CV':
            x_mu_vel_estimate.append(transition_model.mu_W[0,1,0])
            y_mu_vel_estimate.append(transition_model.mu_W[0,1,1])
        print(f"track length ={len(track)} of {len(measurements)}")
    
    from stonesoup.smoother.particle import MarginalisedKalmanSmoother, ParticleSmoother, CarterKohnSmoother
    particlesmoother=ParticleSmoother()
    culled_track=particlesmoother.particle_paths(track=track)
    print('culled')
    RTSsmoother=MarginalisedKalmanSmoother()
    RTS_track=RTSsmoother.smooth(track=track)
    print('RTS done')
    CKsmoother=CarterKohnSmoother()
    # CK_track=CKsmoother.smooth(track= track)
    print('CK done')

In [ ]:
from pathlib import Path
from stonesoup.plotter import Plotterly, Dimension
import numpy as np

# your colour dict
colors_dict = {
    'truth':        'blue',
    'track':        '#00CC96',
    'culled_track': '#FFA15A',
    'RTS_track':    '#B6E880',
    'CK_track':     '#AB63FA'
}

axis_label_list = ["X1(t)","dX1(t)_dt","X2(t)","dX2(t)_dt"]

for i, label in enumerate(axis_label_list):
    # select the right mu data & estimates
    if i <= 1:
        data           = x_mu_data
        data_estimate  = x_mu_estimate
    else:
        data           = y_mu_data
        data_estimate  = y_mu_estimate

    # for CV only, pick velocity
    if mu_model == 'CV':
        if i <= 1:
            vel_data           = x_mu_vel
            vel_data_estimate  = x_mu_vel_estimate
        else:
            vel_data           = y_mu_vel
            vel_data_estimate  = y_mu_vel_estimate

    # -----------------------------------------------------------------
    # 3) Filtering/Smoothing: x dims only (i=0,2)
    # -----------------------------------------------------------------
    if i in (0, 2) and plot_tracks:
        p = new_plotter(label)
        p.plot_ground_truths(truth, [i], mode="lines",
                             line=dict(color=colors_dict['truth'], width=3))
        p.plot_measurements(measurements, [i],
                            marker=dict(symbol="x", size=6))
        p.plot_tracks(track,       [i], mode="lines",
                      uncertainty=uncertainty, particle=particle,
                      plot_particle_paths=plot_particle_paths,
                      track_label="Filtered",
                      line=dict(color=colors_dict['track'], width=3))
        p.plot_tracks(culled_track, [i], mode="lines",
                      uncertainty=uncertainty, particle=particle,
                      plot_particle_paths=plot_particle_paths,
                      track_label="'Descendant'",
                      line=dict(color=colors_dict['culled_track'], width=3))
        p.plot_tracks(RTS_track,   [i], mode="lines",
                      uncertainty=uncertainty, particle=particle,
                      plot_particle_paths=plot_particle_paths,
                      track_label="RTS",
                      line=dict(color=colors_dict['RTS_track'], width=3))
        # p.plot_tracks(CK_track,    [i], mode="lines",
        #               uncertainty=uncertainty, particle=particle,
        #               plot_particle_paths=plot_particle_paths,
        #               track_label="CK",
        #               line=dict(color=colors_dict['CK_track'], width=3))
        p.fig.update_yaxes(gridcolor='lightgrey')
        p.fig.update_layout(legend=dict(y=0.081))
        fname = f"{mu_model}_1D_{label}_filter_smooth.html"
        p.fig.write_html(Path(folder_path)/fname)
        particle_plotter_dict[f"{label}_filter_smooth"] = p

    # -----------------------------------------------------------------
    # 4) Filtering/Smoothing: velocity dims only (i=1,3)
    # -----------------------------------------------------------------
    if i in (1, 3) and plot_tracks:
        p = new_plotter(label)
        p.plot_ground_truths(truth, [i], mode="lines",
                             line=dict(color=colors_dict['truth'], width=3))
        p.plot_tracks(track,       [i], mode="lines",
                      uncertainty=uncertainty, particle=particle,
                      plot_particle_paths=plot_particle_paths,
                      track_label="Filtered",
                      line=dict(color=colors_dict['track'], width=3))
        p.plot_tracks(culled_track, [i], mode="lines",
                      uncertainty=uncertainty, particle=particle,
                      plot_particle_paths=plot_particle_paths,
                      track_label="'Descendant'",
                      line=dict(color=colors_dict['culled_track'], width=3))
        p.plot_tracks(RTS_track,   [i], mode="lines",
                      uncertainty=uncertainty, particle=particle,
                      plot_particle_paths=plot_particle_paths,
                      track_label="RTS",
                      line=dict(color=colors_dict['RTS_track'], width=3))
        # p.plot_tracks(CK_track,    [i], mode="lines",
        #               uncertainty=uncertainty, particle=particle,
        #               plot_particle_paths=plot_particle_paths,
        #               track_label="CK",
        #               line=dict(color=colors_dict['CK_track'], width=3))
        p.fig.update_yaxes(gridcolor='lightgrey')
        p.fig.update_layout(legend=dict(y=0.081))
        fname = f"{mu_model}_1D_{label}_filter_smooth_vel.html"
        p.fig.write_html(Path(folder_path)/fname)
        particle_plotter_dict[f"{label}_filter_smooth_vel"] = p

    # -----------------------------------------------------------------
    # 5) µ ESTIMATE vs REAL (same axis) for x dims only (i=0,2)
    # -----------------------------------------------------------------
    if i in (0, 2):
        p = new_plotter(label)
        # real µ on primary
        p.fig.add_scatter(x=timesteps, y=np.array(data),
                          name='µ true', line=dict(color='green', width=3))
        # estimate µ
        p.fig.add_scatter(x=timesteps, y=np.array(data_estimate),
                          name='µ est',  line=dict(color='pink',  width=3))
        p.fig.add_scatter(x=timesteps, y=np.zeros_like(data),
                          name='0',
                          line=dict(color='gray', dash='dash'))
        p.fig.update_yaxes(title_text='µ̇',gridcolor='lightgrey')
        fname = f"{mu_model}_1D_{label}_mu_compare.html"
        p.fig.write_html(Path(folder_path)/fname)
        particle_plotter_dict[f"{label}_mu_compare"] = p


    # -----------------------------------------------------------------
    # 6) CV only: µ̇ ESTIMATE vs REAL (same axis) for x dims only
    # -----------------------------------------------------------------
    if mu_model == 'CV' and i in (0, 2):
        p=new_plotter(label)
        p.fig.add_scatter(x=timesteps, y=np.array(vel_data),
                          name='µ̇ velocity true', line=dict(color='green', dash='dash', width=3))
        p.fig.add_scatter(x=timesteps, y=np.array(data), secondary_y=True,
                          name='µ̇  true', line=dict(color='red', width=3))
        p.fig.add_scatter(x=timesteps, y=np.zeros_like(data),
                          name='0', secondary_y=True,
                          line=dict(color='gray', dash='dash'))
        p.fig.update_yaxes(title_text='true µ velocity',range=[-np.max(np.abs(vel_data))*1.1,
                                  np.max(np.abs(vel_data))*1.1])
        p.fig.update_yaxes(title_text='true µ position component',secondary_y=True,range=[-np.max(np.abs(data))*1.1,
                                  np.max(np.abs(data))*1.1])
        fname = f"{mu_model}_1D_{label}_mu_vel_compare.html"
        p.fig.write_html(Path(folder_path)/fname)
        particle_plotter_dict[f"{label}_mu_vel_compare"] = p